# Leverage and Cook's distance

Leverage says how unusual an $x$ value is. Cook's distance says how much deleting that row moves the coefficient vector. A point can be unusual in $x$, unusual in its residual, or both.


## Cook's distance

For a fit with $p$ coefficients and residual variance $s^2$, Cook's distance of row $i$ is

$$
D_i = \dfrac{e_i^2}{p\, s^2}\cdot\dfrac{h_{ii}}{(1 - h_{ii})^2}.
$$

It is the squared change in the coefficient vector, scaled by $p s^2$, between the full fit and the fit that deletes row $i$. A rough warning light used in exploration is $D_i > 4/n$. It is not a test.


## The five-point rows

The residuals are $1, -1, 0, -1, 1$, the leverages are $3/5,\ 3/10,\ 1/5,\ 3/10,\ 3/5$, $p = 2$, and $s^2 = 4/3$. Thus $p s^2 = 8/3$, and every nonzero residual has $e_i^2/(p s^2) = 3/8$.

At an endpoint, $h = 3/5$ and

$$
\dfrac{h}{(1-h)^2} = \dfrac{3/5}{(2/5)^2} = \dfrac{15}{4}, \qquad
D = \dfrac{3}{8}\cdot\dfrac{15}{4} = \dfrac{45}{32}.
$$

At the next rows, $h = 3/10$ and

$$
\dfrac{h}{(1-h)^2} = \dfrac{3/10}{(7/10)^2} = \dfrac{30}{49}, \qquad
D = \dfrac{3}{8}\cdot\dfrac{30}{49} = \dfrac{45}{196}.
$$

The middle residual is zero, so its Cook's distance is zero. Since $4/n = 4/5 = 0.8$ and

$$
\dfrac{45}{32} = 1.40625 > 0.8, \qquad \dfrac{45}{196} \approx 0.230 < 0.8,
$$

the warning light marks the two endpoints and not the inner points. Both endpoints have the same distance because they have the same absolute residual and the same leverage.


In [1]:
# Cook's distance for the five residuals of the line 1 + 2x.
from fractions import Fraction
residuals = [1, -1, 0, -1, 1]
leverages = [Fraction(3, 5), Fraction(3, 10), Fraction(1, 5), Fraction(3, 10), Fraction(3, 5)]
p, s2 = 2, Fraction(4, 3)

def cook(error, leverage):
    return (Fraction(error ** 2) / (p * s2)) * (leverage / (1 - leverage) ** 2)

distances = [cook(error, leverage) for error, leverage in zip(residuals, leverages)]
print("Cook's distances", distances)
print("cutoff 4/n", Fraction(4, 5))
assert distances[0] == distances[4] == Fraction(45, 32)
assert distances[1] == distances[3] == Fraction(45, 196)
assert distances[2] == 0
assert distances[0] > Fraction(4, 5) > distances[1]


Cook's distances [Fraction(45, 32), Fraction(45, 196), Fraction(0, 1), Fraction(45, 196), Fraction(45, 32)]
cutoff 4/n 4/5


## Deleting a far point, in both versions

Add the point $(10, 1)$ to the five-point table. The new sums are $n = 6$, $\sum x = 25$, $\sum y = 36$, $\sum xy = 135$, and $\sum x^2 = 155$. Then

$$
\begin{aligned}
b_1 &= \dfrac{6\cdot 135 - 25\cdot 36}{6\cdot 155 - 25^2} = \dfrac{-180}{305} = -\dfrac{18}{61}, \\[4pt]
b_0 &= \dfrac{36}{6} - \left(-\dfrac{18}{61}\right)\dfrac{25}{6} = \dfrac{441}{61}.
\end{aligned}
$$

The slope has changed from $+2$ to $-18/61$. One distant, poorly fitting point turned the line around.

Add $(10, 21)$ instead. The value $21$ lies on $1 + 2x$. The six-point least-squares line remains $1 + 2x$. A far point with a zero residual does not move the coefficients. Distance in $x$ without a residual is leverage without influence. Cook's distance combines the two.


In [2]:
# A far miss flips the slope. The same x on the line leaves the slope at 2.
from fractions import Fraction

def fit(x, y):
    n = len(x)
    sx, sy = sum(x), sum(y)
    sxy = sum(a * b for a, b in zip(x, y))
    sx2 = sum(a * a for a in x)
    slope = Fraction(n * sxy - sx * sy, n * sx2 - sx * sx)
    intercept = Fraction(sy, n) - slope * Fraction(sx, n)
    return intercept, slope

base_x = [1, 2, 3, 4, 5]
base_y = [4, 4, 7, 8, 12]
flipped = fit(base_x + [10], base_y + [1])
quiet = fit(base_x + [10], base_y + [21])
print("with (10, 1)", flipped)
print("with (10, 21)", quiet)
assert flipped == (Fraction(441, 61), Fraction(-18, 61))
assert quiet == (1, 2)


with (10, 1) (Fraction(441, 61), Fraction(-18, 61))
with (10, 21) (Fraction(1, 1), Fraction(2, 1))


## Pitfall

Cook's distance compares coefficient vectors. It does not say that the endpoint is an error in the laboratory book. On the original five-point table the residuals are only $\pm 1$, and the distances $45/32$ are large because $n$ is small and the endpoints already hold leverage $3/5$. Deleting them changes a short fit because a short fit has little else to lean on. Influence is a reason to refit and look, not a reason to discard a row automatically.


In [3]:
# The influential residuals are size 1. Influence is not the same as a huge error.
from fractions import Fraction
print("endpoint residual", 1)
print("endpoint Cook distance", Fraction(45, 32))
assert abs(1) < 2
assert Fraction(45, 32) > 1


endpoint residual 1
endpoint Cook distance 45/32


## Summary

- $D_i$ grows with the squared residual and with $h_{ii}/(1-h_{ii})^2$.
- On the five-point line the endpoint distances are $45/32$ and the inner nonzero distances are $45/196$.
- The added point $(10, 1)$ changes the slope from $2$ to $-18/61$. The point $(10, 21)$, on the line, does not change it.
